# E-Commerce Behavior: Marketing Funnel and Conversion Analysis

## Project Overview
This notebook provides a comprehensive analysis of customer behavior on a large multi-category e-commerce platform. We analyze the full dataset of over 67 million records to understand the customer journey from initial view to final purchase.

### Key Objectives:
1. **Funnel Analysis**: Track user drop-off between View, Cart, and Purchase stages.
2. **Temporal Trends**: Identify peak activity hours and conversion patterns throughout the week.
3. **Price Impact**: Analyze how product pricing influences purchase probability.
4. **Brand & Category Performance**: Identify top-performing brands and product categories.

---

In [10]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import time

# Configuration
INPUT_PATH = "../data/cleaned_data.csv"
CHUNK_SIZE = 2500000
pd.options.plotting.backend = "plotly"

## 1. Data Processing Engine
Given the scale of the dataset (approx. 7.5GB cleaned), we employ a chunked processing strategy to maintain memory efficiency on a 16GB RAM system. We optimize for speed by using string-slicing for temporal features and vectorized aggregations.

In [11]:
# Accumulators
funnel_users = {'view': set(), 'cart': set(), 'purchase': set()}
event_counts = {'view': 0, 'cart': 0, 'purchase': 0}
hourly_users = {i: set() for i in range(24)}
daily_users = {day: set() for day in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']}

price_brackets = ['0-25', '26-50', '51-100', '101-250', '251-500', '501-1000', '1000+']
price_views = {bracket: 0 for bracket in price_brackets}
price_purchases = {bracket: 0 for bracket in price_brackets}

# Brand and Category tracking
brand_events = {}
brand_purchases = {}
category_views = {}

total_users_set = set()
total_rows = 0
start_time = time.time()
date_to_day_cache = {}

print(f"Processing dataset chunks...")

use_cols = ['event_time', 'event_type', 'user_id', 'price', 'brand', 'category_code']

for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNK_SIZE, usecols=use_cols):
    # Temporal Optimization
    chunk['h'] = chunk['event_time'].str[11:13].astype(np.int8)
    dates = chunk['event_time'].str[0:10]
    for d in dates.unique():
        if d not in date_to_day_cache: date_to_day_cache[d] = pd.to_datetime(d).day_name()
    chunk['d'] = dates.map(date_to_day_cache)
    
    # Funnel Updates
    total_users_set.update(chunk['user_id'].unique())
    for etype in ['view', 'cart', 'purchase']:
        mask = chunk['event_type'] == etype
        funnel_users[etype].update(chunk.loc[mask, 'user_id'].unique())
        event_counts[etype] += mask.sum()
    
    # Temporal Groups
    for h, ids in chunk.groupby('h')['user_id'].unique().items(): hourly_users[h].update(ids)
    for d, ids in chunk.groupby('d')['user_id'].unique().items(): daily_users[d].update(ids)
    
    # Price Analysis
    chunk['pr'] = pd.cut(chunk['price'], bins=[0, 25, 50, 100, 250, 500, 1000, float('inf')], labels=price_brackets)
    v_vc = chunk.loc[chunk['event_type'] == 'view', 'pr'].value_counts()
    p_vc = chunk.loc[chunk['event_type'] == 'purchase', 'pr'].value_counts()
    for br in price_brackets:
        if br in v_vc: price_views[br] += v_vc[br]
        if br in p_vc: price_purchases[br] += p_vc[br]
        
    # Brand & Category Tracking
    be = chunk['brand'].value_counts()
    for b, count in be.items(): brand_events[b] = brand_events.get(b, 0) + count
    
    bp = chunk[chunk['event_type'] == 'purchase']['brand'].value_counts()
    for b, count in bp.items(): brand_purchases[b] = brand_purchases.get(b, 0) + count
    
    cv = chunk[chunk['event_type'] == 'view']['category_code'].value_counts()
    for c, count in cv.items(): category_views[c] = category_views.get(c, 0) + count

    total_rows += len(chunk)
    # Professional print without emojis
    print(f"  Progress: {total_rows/1_000_000:.1f}M rows processed...")

print(f"\nAnalysis complete! Total rows: {total_rows:,} | Total time: {(time.time() - start_time)/60:.2f} minutes.")

Processing dataset chunks...
  Progress: 2.5M rows processed...
  Progress: 5.0M rows processed...
  Progress: 7.5M rows processed...
  Progress: 10.0M rows processed...
  Progress: 12.5M rows processed...
  Progress: 15.0M rows processed...
  Progress: 17.5M rows processed...
  Progress: 20.0M rows processed...
  Progress: 22.5M rows processed...
  Progress: 25.0M rows processed...
  Progress: 27.5M rows processed...
  Progress: 30.0M rows processed...
  Progress: 32.5M rows processed...
  Progress: 35.0M rows processed...
  Progress: 37.5M rows processed...
  Progress: 40.0M rows processed...
  Progress: 42.5M rows processed...
  Progress: 45.0M rows processed...
  Progress: 47.5M rows processed...
  Progress: 50.0M rows processed...
  Progress: 52.5M rows processed...
  Progress: 55.0M rows processed...
  Progress: 57.5M rows processed...
  Progress: 60.0M rows processed...
  Progress: 62.5M rows processed...
  Progress: 65.0M rows processed...
  Progress: 67.4M rows processed...

A

## 2. Marketing Funnel Visualization
The purchase funnel reveals how many unique users transition through the customer journey.

In [12]:
fig_funnel = go.Figure(go.Funnel(
    y = ["Product Views", "Add to Carts", "Confirmed Purchases"],
    x = [len(funnel_users[s]) for s in ['view', 'cart', 'purchase']],
    textinfo = "value+percent initial",
    marker = {"color": ["royalblue", "darkorange", "forestgreen"]}))

fig_funnel.update_layout(title_text="User Conversion Funnel", template="plotly_white")
fig_funnel.show()
fig_funnel.write_image("../screenshots/1_marketing_funnel.png")

## 3. Temporal Engagement Trends
Understanding when users are most active allows for better marketing campaign timing.

In [13]:
hour_labels = [f"{i:02d}:00" for i in range(24)]
hour_vals = [len(hourly_users[i]) for i in range(24)]
fig_h = px.line(x=hour_labels, y=hour_vals, title="Hourly User Activity", labels={'x':'Time of Day','y':'Unique Users'})
fig_h.show()

fig_h.write_image("../screenshots/2_hourly_activity.png")
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = [len(daily_users[d]) for d in day_order]
fig_d = px.bar(x=day_order, y=day_counts, title="Daily Activity Distribution", labels={'x':'Day','y':'Users'}, color=day_counts)
fig_d.show()
fig_d.write_image("../screenshots/3_daily_activity.png")

## 4. Price Sensitivity and Conversion Performance
We analyze how different price points influence user conversion. This helps identify the 'sweet spot' for pricing strategy on the platform.

In [14]:
# 4.1 Conversion Rate by Price Bracket
pb_cr = {b: (price_purchases.get(b,0) / price_views.get(b,1) * 100) if price_views.get(b,1) > 0 else 0 for b in price_brackets}
fig_pb_cr = px.bar(x=list(pb_cr.keys()), y=list(pb_cr.values()), 
                   title="Purchase Conversion Rate by Price Bracket (%)",
                   labels={'x':'Price Range ($)','y':'Conversion Ratio (%)'},
                   color=list(pb_cr.values()),
                   color_continuous_scale='Turbo')
fig_pb_cr.update_layout(template='plotly_white', showlegend=False)
fig_pb_cr.show()

fig_pb_cr.write_image("../screenshots/4_price_bracket_cr.png")
# 4.2 Purchase Volume Distribution by Price
fig_pb_dist = px.pie(names=list(price_purchases.keys()), values=list(price_purchases.values()), 
                    title="Distribution of Confirmed Purchases across Price Brackets",
                    hole=0.4,
                    color_discrete_sequence=px.colors.sequential.RdBu)
fig_pb_dist.update_traces(textinfo='percent+label')
fig_pb_dist.show()
fig_pb_dist.write_image("../screenshots/5_purchase_price_dist.png")

## 5. Advanced Behavioral Insights
This section explores the broader engagement metrics, including the distribution of event types and temporal consistency of user activity.

In [15]:
# 5.1 Overall Event Distribution
event_df = pd.Series(event_counts).reset_index()
event_df.columns = ['Event Type', 'Count']
fig_events = px.pie(event_df, values='Count', names='Event Type', 
                    title='Global Event Distribution (Views vs. Carts vs. Purchases)',
                    hole=0.5, 
                    color_discrete_sequence=['#636EFA', '#EF553B', '#00CC96'])
fig_events.update_traces(textinfo='percent+label')
fig_events.update_layout(template='plotly_white')
fig_events.show()

fig_events.write_image("../screenshots/6_event_distribution.png")
# 5.2 Weekly Activity Intensity
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_active = pd.Series({d: len(daily_users.get(d, [])) for d in days_order})
fig_daily = px.line(x=daily_active.index, y=daily_active.values, 
                   title="User Engagement Intensity by Day of the Week",
                   labels={'x':'Day of Week','y':'Total Unique Users'},
                   markers=True)
fig_daily.update_traces(line_color='#FF6692', fill='tozeroy')
fig_daily.update_layout(template='plotly_white')
fig_daily.show()
fig_daily.write_image("../screenshots/7_weekly_active_users.png")

## 6. Market Performance: Brands & Categories
Analysis of brand loyalty and category-level interest to identify high-growth opportunities.

In [16]:
# 6.1 Brand Market Dominance
top_brands = pd.Series(brand_events).sort_values(ascending=False).head(10)
fig_b = px.bar(x=top_brands.index, y=top_brands.values, 
               title="Top 10 Brands by Total Engagement (View + Cart + Purchase)", 
               labels={'x':'Brand','y':'Total Event Volume'},
               color=top_brands.values,
               color_continuous_scale='Viridis')
fig_b.update_layout(template='plotly_white')
fig_b.show()

fig_b.write_image("../screenshots/8_brand_engagement.png")
# 6.2 Category Interest Distribution
top_cats = pd.Series(category_views).sort_values(ascending=False).head(10)
fig_c = px.pie(names=top_cats.index, values=top_cats.values, 
               title="Category Interest Heatmap (by View Volume)",
               hole=0.4,
               color_discrete_sequence=px.colors.qualitative.Pastel)
fig_c.update_layout(template='plotly_white')
fig_c.show()

fig_c.write_image("../screenshots/9_category_interest.png")
# 6.3 Conversion Efficiency Index (Top 12 Brands)
top_purch_brands = pd.Series(brand_purchases).sort_values(ascending=False).head(12).index
brand_cr = {b: (brand_purchases.get(b,0) / brand_events.get(b,1) * 100) for b in top_purch_brands if b != 'unknown'}
brand_cr_sorted = pd.Series(brand_cr).sort_values(ascending=False)
fig_bcr = px.bar(x=brand_cr_sorted.index, y=brand_cr_sorted.values, 
                 title="Brand Conversion Efficiency Index (%)", 
                 labels={'x':'Brand Name','y':'Conversion Efficiency (%)'},
                 color=brand_cr_sorted.values,
                 color_continuous_scale='Cividis')
fig_bcr.update_layout(template='plotly_white')
fig_bcr.show()
fig_bcr.write_image("../screenshots/10_brand_conversion_efficiency.png")

## 7. Executive Summary & Actionable Insights

Based on the analysis of **67.4 million** behavior records, we have identified several critical optimization points for the platform:

1. **Funnel Friction**: The most significant user drop-off occurs between the `view` and `cart` stages. Marketing efforts should focus on 'retargeting' users who viewed high-value products without adding them to their cart.
2. **Timing Strategy**: User activity peaks between **13:00 and 16:00**. Flash sales and inventory updates should be synchronized with this window to maximize visibility.
3. **Price Positioning**: Products in the **101-250$** range demonstrate the highest conversion efficiency, indicating a strong 'mid-tier' market segment that delivers consistent ROI.
4. **Brand Loyalty**: Brands like **Samsung** and **Apple** dominate the engagement metrics, but niche brands often show higher conversion efficiency (CR%). This suggests that while mainstream brands drive traffic, specialty brands drive profit.
5. **Temporal Consistency**: Engagement remains relatively stable throughout the week, with a slight uptick on **weekends**, suggesting that the platform has a loyal, daily-active user base.